In [1]:
import pandas as pd 
import sklearn as sk
import os
import pandas as pd
import numpy as np 
import sklearn
import matplotlib.pyplot as plt
import seaborn as sns 
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import RobustScaler
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix



Matplotlib is building the font cache; this may take a moment.


In [ ]:
csv_path='NT_GNN_vanilla/'
grafo_path='/mnt/trcanmed/snaketree/stash/degK_not.tsv'

#egrassi@godot:/scratch/trcanmed/DE_RNASeq/dataset/scRNA_deg$ cat KRAS_cutoff0.05-KRAS.vs.WT.goinsplit_down.tsv KRAS_cutoff0.05-KRAS.vs.WT.goinsplit_up.tsv > /mnt/trcanmed/snaketree/stash/degK.tsv

grafo=pd.read_csv(grafo_path)
geni=grafo['gene'].unique()

In [ ]:
train_id=['filtered_CRC0327_NT_2.csv','filtered_CRC0542_NT72h_1.csv','filtered_CRC1620_NT_1.csv','filtered_CRC1139_NT_1.csv']
test_id=['filtered_CRC0322_NT_1_3000.csv','filtered_CRC1502_NT_1.csv']
kras=['CRC1502','CRC1620','CRC1139']
wt=['CRC0322','CRC0327','CRC0542']
train=pd.DataFrame()
test=pd.DataFrame()
for file in os.listdir(csv_path):
    sample_name=str.split(file,sep='_')[1]
    data=pd.read_csv(os.path.join(csv_path,file),header=0,index_col=0)
    valid_geni=[g for g in geni if g in data.columns]
    data=data.loc[:,valid_geni]
    data['sample']=sample_name
    data['cell_id']=data.index
    data.reset_index(drop=True,inplace=True)
    if sample_name in kras:
        data['label']=1
    else:
        data['label']=0
    if file in train_id:
        print('train') 
        train=pd.concat([train,data])
    else:
        print('test')
        test=pd.concat([test,data])
    

In [3]:
geni_senza_na = train.columns[train.isnull().sum() == 0]
print(f"Geni senza NaN nel train: {len(geni_senza_na)}")
geni_senza_na_test = test.columns[test.isnull().sum() == 0]
print(f"Geni senza NaN nel test: {len(geni_senza_na_test)}")

geni = [g for g in geni if g in train.columns]
geni_validi = [
    g for g in geni
    if g in train.columns and g in test.columns and
    train[g].isnull().sum() == 0 and test[g].isnull().sum() == 0
]

#filtro geni con na
meta_col = [c for c in ['label', 'sample', 'cell_id'] if c in train.columns]
train = train[geni_validi + meta_col]
test = test[geni_validi + meta_col]

NameError: name 'train' is not defined

In [ ]:
#separo metadati e dato numerico
Y_train=train['label']
X_train=train[geni_validi]
Y_test=test['label']
X_test=test[geni_validi]

In [ ]:
#standardizzo e faccio PCA
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

pca = PCA()
X_train_pca = pca.fit_transform(X_train_scaled)

# Varianza spiegata da ogni componente
explained_var = pca.explained_variance_ratio_
cumulative_var = explained_var.cumsum()
#varianza spiegata asse y max 1
plt.figure(figsize=(8,5))
plt.plot(range(1, len(explained_var)+1), explained_var, marker='o')
plt.title('Varianza spiegata da ciascuna componente PCA')
plt.xlabel('Numero componente')
plt.ylabel('Varianza spiegata')
plt.grid(True)
plt.show()

#varianza cumulativa
cumulative_var = np.cumsum(pca.explained_variance_ratio_)

plt.figure(figsize=(8,5))
plt.plot(range(1, len(cumulative_var)+1), cumulative_var, marker='o')
plt.axhline(y=0.90, color='r', linestyle='--', label='90% varianza')
plt.axhline(y=0.95, color='g', linestyle='--', label='95% varianza')
plt.xlabel('Numero componente')
plt.ylabel('Varianza cumulativa')
plt.title('Varianza cumulativa (PCA)')
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
import numpy as np


n_components_90 = np.argmax(cumulative_var >= 0.90) + 1

print(f"Componenti per spiegare ≥90% della varianza: {n_components_90}")


In [ ]:
import numpy as np

# Verifica se ci sono NaN totali
print("Ci sono NaN nel test standardizzato?", np.isnan(X_train_scaled).any())

# Quante colonne contengono NaN
na_col_mask = np.isnan(X_train_scaled).any(axis=0)
print(f"Numero di colonne con NaN: {np.sum(na_col_mask)}")

In [ ]:
pca = PCA(n_components=n_components_90)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca=pca.transform(X_test_scaled)

In [ ]:
import matplotlib.pyplot as plt

train_pca_df = pd.DataFrame(
    X_train_pca[:,[0,1]],
    columns=["PC1", "PC2"]
)
train_pca_df["label"] = Y_train.values
train_pca_df["sample"] = train["sample"].values

# Plot
plt.figure(figsize=(8,6))
for lbl, color in zip([0,1], ["blue", "red"]):
    subset = train_pca_df[train_pca_df["label"] == lbl]
    plt.scatter(subset["PC1"], subset["PC2"],
                c=color, label=f"KRAS={lbl}", alpha=0.3, s=10)

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA: PC1 vs PC2 colorato per label")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
#SVM 
param_grid_rbf = {
    'C': [0.01, 0.1, 1, 5],
    'gamma': [0.001, 0.01, 0.1, 1, 'scale']
}

svm_rbf = SVC(kernel='rbf')
grid_rbf = GridSearchCV(svm_rbf, param_grid_rbf, cv=3, scoring='f1', verbose=3,n_jobs=3)
grid_rbf.fit(X_train_pca, Y_train)




In [ ]:
print("Migliori parametri trovati:", grid_rbf.best_params_)
print(f"Score migliore in CV: {grid_rbf.best_score_:.3f}")

In [ ]:
best_model = grid_rbf.best_estimator_
y_pred = best_model.predict(X_test_pca)

print("\nReport test set:")
print(classification_report(Y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(Y_test, y_pred))
